---
# 🚇 Urban Growth Dynamics of ASEAN Megacities
## Through the Lens of Public Transport Ridership
### การวิเคราะห์การเติบโตของมหานครอาเซียนผ่านเลนส์ระบบขนส่งสาธารณะ
---
## 📋 Executive Summary — บทสรุปผู้บริหาร

&emsp;ลองนึกภาพตามนะคะ — ทุกเช้า คนหลายสิบล้านคนในอาเซียนออกจากบ้าน บางคนขึ้น BTS บางคนนั่ง MRT สิงคโปร์ บางคนต่อรถเมล์ในกัวลาลัมเปอร์ การเคลื่อนไหวของคนเหล่านี้ไม่ใช่แค่ "การเดินทาง" — มันคือ **สัญญาณชีพ** ของเมือง

&emsp;เมื่อเศรษฐกิจดี คนออกไปทำงาน ช้อปปิ้ง พบปะสังสรรค์ — ตัวเลขผู้โดยสารพุ่งขึ้น เมื่อเกิดวิกฤต เช่น COVID-19 — ตัวเลขดิ่งลงในชั่วข้ามคืน นี่คือสิ่งที่รายงานฉบับนี้ต้องการพิสูจน์

| ส่วน | เมือง/หัวข้อ | สิ่งที่จะค้นพบ |
|------|-------------|----------------|
| **Part 1** | กรุงเทพฯ 🇹🇭 | คนกรุงเทพฯ เดินทางด้วยอะไร เพิ่มขึ้นแค่ไหน? |
| **Part 2** | สิงคโปร์ 🇸🇬 | ระบบขนส่งระดับโลกเขาทำได้อย่างไร? |
| **Part 3** | กัวลาลัมเปอร์ 🇲🇾 | มาเลย์กำลังไปในทิศทางไหน? |
| **Part 4** | จาการ์ตา 🇮🇩 | เมืองที่ใหญ่ที่สุดในอาเซียนอยู่ตรงไหน? |
| **Part 5** | มะนิลา 🇵🇭 | ฟิลิปปินส์โตมา 25 ปีอย่างไร? |
| **Part 6** | Economic Growth | ขนส่งกับเศรษฐกิจ — มีความสัมพันธ์กันจริงไหม? |
| **Part 7** | Development Gap | เมืองกำลังพัฒนาตามสิงคโปร์ทัน? |

---

## 🗂️ ที่มาและความสำคัญ

&emsp;อาเซียนมีประชากรกว่า **650 ล้านคน** กระจายอยู่ใน 10 ประเทศ และกำลังก้าวเข้าสู่ยุคของการขยายตัวอย่างรวดเร็ว เมืองใหญ่ๆ เช่น กรุงเทพฯ จาการ์ตา และมะนิลา ต่างแบกรับประชากรหลักสิบล้านคน พร้อมกับความท้าทายด้านการจราจรและมลพิษที่ตามมา

&emsp;ระบบขนส่งสาธารณะจึงไม่ใช่แค่ "รถไฟฟ้า" หรือ "รถเมล์" — มันคือตัวชี้วัดว่าเมืองนั้น **พัฒนาไปถึงไหน** แล้ว

**ข้อมูลที่ใช้วิเคราะห์:**
- 📊 จำนวนผู้โดยสารรายเดือน จำแนกตามระบบขนส่ง (2019–2025)
- 💰 GDP รายเมือง (Billion USD)
- 👥 จำนวนประชากร (ล้านคน)
- 🌫️ ค่าฝุ่น PM2.5 (μg/m³)

---

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── Dark Theme ────────────────────────────────────────────────────────────────
PAPER_BG   = '#0D1117'
PLOT_BG    = '#161B22'
GRID_C     = '#30363D'
FONT_C     = '#E6EDF3'
MUTED_C    = '#8B949E'
TEMPLATE   = 'plotly_dark'

CITY_COLORS = {
    'Bangkok':      '#FF6B6B',
    'Singapore':    '#74B9FF',
    'Kuala Lumpur': '#4ECDC4',
    'Jakarta':      '#FFB347',
    'Manila':       '#C77DFF',
}

def base_layout(height=460, legend_h=True):
    leg = dict(bgcolor='rgba(22,27,34,0.9)', bordercolor=GRID_C, borderwidth=1, font_size=11)
    if legend_h:
        leg.update(orientation='h', y=1.13, x=1, xanchor='right')
    return dict(
        template=TEMPLATE, paper_bgcolor=PAPER_BG, plot_bgcolor=PLOT_BG,
        font=dict(color=FONT_C, family='Segoe UI, Arial, sans-serif'),
        height=height, legend=leg,
        margin=dict(l=60, r=40, t=75, b=55),
        hoverlabel=dict(bgcolor='#1F2937', bordercolor=GRID_C, font_size=12),
    )

def ax_style():
    return dict(gridcolor=GRID_C, zerolinecolor=GRID_C)

print('✅ Setup complete — Dark theme loaded')

In [ ]:
# ── Load Data ─────────────────────────────────────────────────────────────────
df = pd.read_csv('ASEAN_Urban_Growth_Final_with_Mode.csv')
df['Date']  = pd.to_datetime(df['Date'])
df['Year']  = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

# ── Mode label maps ───────────────────────────────────────────────────────────
MODE_BKK = {'BTS':'BTS Skytrain','MRT':'MRT Blue/Purple','SRT':'SRT Red Line',
             'ARL':'Airport Rail Link','YL':'MRT Yellow Line','PK':'MRT Pink Line','RL':'Regional Rail'}
MODE_SGP = {'MRT':'MRT','Public Bus':'Public Bus','LRT':'LRT'}
MODE_KL  = {
    'rail_lrt_kj':'LRT Kelana Jaya','rail_mrt_kajang':'MRT Kajang',
    'rail_lrt_ampang':'LRT Ampang','bus_rkl':'RapidKL Bus',
    'rail_mrt_pjy':'MRT Putrajaya','rail_monorail':'KL Monorail',
    'rail_komuter':'KTM Komuter','rail_komuter_utara':'KTM Utara',
    'rail_ets':'ETS Train','rail_intercity':'Intercity Rail',
    'rail_tebrau':'KTM Tebrau','bus_rkn':'RapidKN Bus','bus_rpn':'RapidPN Bus'
}
MODE_JKT = {'TRANSJAKARTA':'TransJakarta (BRT)','KRL':'KRL Commuter',
             'MRT':'MRT Jakarta','LRT':'LRT Jakarta',
             'BUS SEKOLAH':'School Bus','KCI COMMUTER BANDARA':'Airport Rail','KAPAL':'Ferry'}

print(f'Dataset: {df.shape[0]:,} rows | {df["City"].nunique()} cities')
print('Cities:', df['City'].unique().tolist())

---
## Part 6 — Economic Growth 💹
### ขนส่งสาธารณะกับเศรษฐกิจ — มีความสัมพันธ์กันจริงไหม?

&emsp;ลองคิดแบบนี้ — เมื่อคนมีงานทำ มีเงินในกระเป๋า พวกเขาจะออกไปข้างนอกมากขึ้น ไปห้างสรรพสินค้า ไปทำงาน ไปพบเพื่อน ทุกการเดินทางนั้น ถ้าใช้ระบบขนส่งสาธารณะ มันจะปรากฏอยู่ในตัวเลข Ridership

&emsp;ในทางกลับกัน เมื่อเศรษฐกิจหดตัว คนก็อยู่บ้านมากขึ้น ตัวเลขก็ลดลง

&emsp;ส่วนนี้จะทดสอบความสัมพันธ์นี้ผ่านข้อมูลจริงจาก 5 เมือง ผ่าน 3 กราฟ: แนวโน้มรายปี, อัตราการเปลี่ยนแปลง YoY และความสัมพันธ์กับ GDP

---

In [ ]:
# ── Econ prep ─────────────────────────────────────────────────────────────────
yearly = df.groupby(['Year','City'])['Ridership'].sum().reset_index(name='Total_Ridership')
gdp_yr = df.groupby(['Year','City']).agg(
    Total_Ridership=('Ridership','sum'), GDP=('GDP_Billion_USD','mean')
).reset_index()

# ── Chart 6.1: Multi-city Ridership Trend ────────────────────────────────────
c1 = yearly[yearly['Year'].between(2019,2025)]
fig = px.line(c1, x='Year', y='Total_Ridership', color='City',
    markers=True, line_shape='spline',
    title='<b>ตารางที่ 6.1</b>  แนวโน้มผู้โดยสารรายปี ทั้ง 5 เมือง (2019–2025)',
    labels={'Total_Ridership':'ผู้โดยสารรวม','Year':''},
    color_discrete_map=CITY_COLORS)
fig.add_vrect(x0=2019.8, x1=2021.2, fillcolor='#FF4D4D', opacity=0.07,
              annotation_text='⚠ COVID-19', annotation_position='top left',
              annotation_font_color='#FF8080', annotation_font_size=11)
fig.update_layout(**base_layout(480), hovermode='x unified',
                  xaxis=dict(dtick=1,**ax_style()),
                  yaxis=dict(tickformat='.2s',**ax_style()))
fig.update_traces(line_width=2.5, marker_size=7)
fig.show()

# ── Chart 6.2: YoY All Cities ────────────────────────────────────────────────
yoy_all=[]
for city in yearly['City'].unique():
    sub = yearly[yearly['City']==city].sort_values('Year').copy()
    sub['YoY'] = sub['Total_Ridership'].pct_change()*100
    yoy_all.append(sub)
yoy_df = pd.concat(yoy_all).dropna(subset=['YoY'])
yoy_df = yoy_df[yoy_df['Year'].between(2020,2025)]

fig = px.bar(yoy_df, x='Year', y='YoY', color='City', barmode='group',
    title='<b>ตารางที่ 6.2</b>  อัตราการเปลี่ยนแปลง YoY (%) ทั้ง 5 เมือง',
    labels={'YoY':'YoY Change (%)','Year':''},
    color_discrete_map=CITY_COLORS)
fig.add_hline(y=0, line_dash='dash', line_color=MUTED_C, line_width=1.2)
fig.update_layout(**base_layout(480), hovermode='x unified',
                  xaxis=dict(dtick=1,**ax_style()),
                  yaxis=dict(ticksuffix='%',**ax_style()))
fig.show()

# ── Chart 6.3: Ridership vs GDP Scatter ──────────────────────────────────────
g3 = gdp_yr[gdp_yr['Year'].between(2021,2024)]
fig = px.scatter(g3, x='GDP', y='Total_Ridership',
    color='City', size='Total_Ridership', size_max=45,
    hover_data=['Year'], text='Year',
    trendline='ols', trendline_scope='trace',
    title='<b>ตารางที่ 6.3</b>  ความสัมพันธ์ระหว่างผู้โดยสารกับ GDP (2021–2024)',
    labels={'GDP':'GDP (Billion USD)','Total_Ridership':'ผู้โดยสารรวม'},
    color_discrete_map=CITY_COLORS)
fig.update_traces(textposition='top center', textfont_size=9,
                  selector=dict(mode='markers+text'))
fig.update_layout(**base_layout(520),
                  xaxis=dict(tickprefix='$',ticksuffix='B',**ax_style()),
                  yaxis=dict(tickformat='.2s',**ax_style()))
fig.show()

> 📌 **สรุป Part 6 — Economic Growth:**
>
> ภาพที่เห็นชัดที่สุดคือ **ทุกเมืองเดินไปพร้อมกัน** — เมื่อ COVID ระบาดในปี 2020 ตัวเลขผู้โดยสารของทุกเมืองดิ่งพร้อมกัน แล้วก็ฟื้นขึ้นมาพร้อมกันในปี 2022 เหมือนกับที่คลื่นขึ้นและลงพร้อมกัน นี่เป็นหลักฐานที่ชัดเจนว่า **ระบบขนส่งสะท้อนการเต้นของชีพจรเศรษฐกิจได้จริง**
>
> Scatter Chart ตารางที่ 6.3 ยืนยันว่า **เมืองที่มี GDP สูงกว่า มีแนวโน้มมีผู้โดยสารมากกว่า** อย่างมีนัยสำคัญ โดยเส้น Trendline ในทุกเมืองมีความชันเป็นบวก Singapore เป็นตัวอย่างที่ดีที่สุด — GDP สูง, Ridership สูง และทั้งสองเติบโตไปด้วยกัน

---
## Part 7 — Development Gap 🌏
### เมืองกำลังพัฒนาตามสิงคโปร์ทัน?

&emsp;สิงคโปร์เป็นเมืองที่มีประชากรเพียง 5.9 ล้านคน แต่กลับมีผู้โดยสารระบบขนส่งปีละกว่า **2,700 ล้านเที่ยว** นั่นหมายความว่า คนสิงคโปร์ 1 คน ใช้ระบบขนส่งสาธารณะเฉลี่ยกว่า **450 เที่ยวต่อปี** หรือแทบทุกวัน

&emsp;เปรียบเทียบกับกรุงเทพฯ ที่มีคน 10 ล้านคน แต่มีผู้โดยสารเพียง ~540 ล้านเที่ยวต่อปี หรือประมาณ **50 เที่ยวต่อคนต่อปี** — ห่างกัน 9 เท่า!

&emsp;คำถามคือ ช่องว่างนี้กำลังแคบลงหรือไม่?

---